# GSVC-Turbo — PRD Ablation Test Suite

Four ablation experiments from PRD §5:

| Test | What runs | Purpose |
|------|-----------|---------|
| **T1** Baseline | `train_video_Represent.py` (no `--turbo`) | Establish reference BPP/PSNR/time |
| **T2** Full Turbo | All FR1-FR7 enabled (`--turbo`) | Measure iteration reduction & RD gain |
| **T3** Clamp Ablation | `--turbo_preset ablation_no_clamp` | Track PSNR degradation without FR3+FR7 |
| **T4** VRAM Ablation | `--turbo_preset ablation_no_downscale` | Document VRAM spike without FR1 |

Run cells in order: **Setup → T1 → T2 → T3 → T4 → Compare**.


## Setup — Clone, Install, Write Turbo Files

In [ ]:
# ============================================================
# SETUP: Clone GSVC, install deps, write turbo module files
# ============================================================
import re, os
from pathlib import Path

!git clone -b d https://github.com/VCoklat/GSVC.git
%cd /kaggle/working/GSVC

# Normalize any accidentally-indented turbo config functions
config_path = Path("config_v2.py")
if config_path.exists():
    txt = config_path.read_text()
    txt = re.sub(r"^\s+def get_turbo4([1-8])_compress_config",
                 r"def get_turbo4\1_compress_config", txt, flags=re.MULTILINE)
    config_path.write_text(txt)

%cd gsplat
!sed -i 's/pyyaml==6.0/pyyaml>=6.0/g' setup.py
!pip install "pyyaml>=6.0" -q
!pip install .[dev] --no-build-isolation -q
%cd ..
!pip install -r reqirements.txt -q

# Verify RAFT-Small is available (pre-installed on Kaggle GPU)
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
print("torchvision RAFT-Small: OK")

# Write turbo_config.py
Path("/kaggle/working/GSVC/turbo_config.py").write_text("""\
\"\"\"
GSVC-Turbo Configuration
========================
Dataclass-based configuration for the OpticalFlowWarper module.
Controls VRAM protection, flow clamping, and ablation toggles.
\"\"\"

from dataclasses import dataclass, field


@dataclass
class TurboConfig:
    \"\"\"Configuration for the GSVC-Turbo optical flow warping pipeline.\"\"\"

    # ── Master toggle ──────────────────────────────────────────────────
    enabled: bool = True

    # ── FR1: Adaptive VRAM Protection ──────────────────────────────────
    max_resolution: int = 540
    \"\"\"Maximum height (in pixels) fed to RAFT. Frames taller than this
    are bilinearly downsampled before flow estimation. Set to 0 to
    disable downscaling (VRAM Limit Ablation).\"\"\"

    # ── FR2: RAFT Model Selection ──────────────────────────────────────
    raft_model: str = "raft_small"
    \"\"\"Which RAFT variant to use.  Currently only 'raft_small' is
    supported via torchvision.\"\"\"

    raft_iterations: int = 12
    \"\"\"Number of RAFT iterative update steps. More = smoother flow,
    but slower.  12 is a good default for 540p.\"\"\"

    # ── FR3: Flow Clamping ─────────────────────────────────────────────
    max_flow_magnitude: float = 80.0
    \"\"\"Maximum allowed optical flow displacement in pixels (at the
    *original* resolution).  Prevents explosive advection from noisy
    flow in textureless regions. Set to 0 to disable (Clamping Ablation).\"\"\"

    # ── FR7: Output Clamping ───────────────────────────────────────────
    position_clamp: float = 0.999
    \"\"\"Clamp advected normalized positions to [-clamp, +clamp] before
    applying atanh.  Prevents infinity from boundary saturation.
    Set to 1.0 to disable (Clamping Ablation).\"\"\"

    # ── Performance ────────────────────────────────────────────────────
    use_fp16_flow: bool = True
    \"\"\"Run RAFT inference in float16 to reduce VRAM.  Generally safe
    since flow vectors don't need full float32 precision.\"\"\"

    # ── Logging ────────────────────────────────────────────────────────
    verbose: bool = False
    \"\"\"Print per-frame flow statistics (mean displacement, clamped %).\"\"\"


# ── Preset Configurations ──────────────────────────────────────────────

TURBO_PRESETS = {
    "default": TurboConfig(),

    "conservative": TurboConfig(
        max_resolution=720,
        max_flow_magnitude=40.0,
        raft_iterations=20,
        position_clamp=0.995,
    ),

    "aggressive": TurboConfig(
        max_resolution=480,
        max_flow_magnitude=120.0,
        raft_iterations=8,
        position_clamp=0.999,
    ),

    # ── Ablation Presets ───────────────────────────────────────────────
    "ablation_no_clamp": TurboConfig(
        max_flow_magnitude=0.0,   # FR3 disabled
        position_clamp=1.0,       # FR7 disabled
    ),

    "ablation_no_downscale": TurboConfig(
        max_resolution=0,         # FR1 disabled — native resolution
    ),

    "disabled": TurboConfig(enabled=False),
}


def get_turbo_config(preset: str = "default") -> TurboConfig:
    \"\"\"Retrieve a TurboConfig by preset name.\"\"\"
    if preset not in TURBO_PRESETS:
        raise ValueError(
            f"Unknown turbo preset '{preset}'. "
            f"Available: {list(TURBO_PRESETS.keys())}"
        )
    return TURBO_PRESETS[preset]

""")

# Write optical_flow_warper.py
Path("/kaggle/working/GSVC/optical_flow_warper.py").write_text("""\
\"\"\"
GSVC-Turbo — Optical Flow Warper
=================================
Implements the complete FR1-FR7 pipeline for Hybrid Eulerian-Lagrangian
tracking of 2D Gaussian Splatting particles.

Pipeline Summary:
    FR1: Adaptive VRAM Protection (downscale large frames for RAFT)
    FR2: Eulerian Flow Extraction (RAFT-Small dense optical flow)
    FR3: Resolution Restoration & Safety Clamping
    FR4: Particle Space Normalization (tanh transform)
    FR5: Sub-Pixel Eulerian-to-Lagrangian Mapping (grid_sample)
    FR6: Kinematic Advection (translate particles by sampled flow)
    FR7: State Injection & Optimizer Prep (atanh inverse + clamp)

Usage:
    warper = OpticalFlowWarper(device='cuda:0')
    warper.warp(prev_frame, curr_frame, gaussian_model)
\"\"\"

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from turbo_config import TurboConfig

# Lazy-loaded to avoid import overhead when turbo is disabled
_raft_model = None
_raft_device = None


def _get_raft_model(device: torch.device):
    \"\"\"Lazy-load and cache the RAFT-Small model from torchvision.\"\"\"
    global _raft_model, _raft_device
    if _raft_model is not None and _raft_device == device:
        return _raft_model

    from torchvision.models.optical_flow import raft_small, Raft_Small_Weights

    _raft_model = raft_small(weights=Raft_Small_Weights.DEFAULT).to(device)
    _raft_model.eval()
    _raft_device = device

    # Freeze all parameters — we never train RAFT
    for p in _raft_model.parameters():
        p.requires_grad_(False)

    return _raft_model


class OpticalFlowWarper:
    \"\"\"
    Pre-advects Gaussian splat positions using dense optical flow,
    implementing the complete FR1-FR7 Turbo pipeline.

    Args:
        config: TurboConfig instance controlling all pipeline parameters.
        device: torch device for computation (default: cuda:0).
    \"\"\"

    def __init__(
        self,
        config: TurboConfig = None,
        device: torch.device = torch.device("cuda:0"),
    ):
        self.config = config or TurboConfig()
        self.device = device

        # Pre-warm RAFT on first instantiation
        if self.config.enabled:
            _get_raft_model(self.device)

    # ═══════════════════════════════════════════════════════════════════
    # PUBLIC API
    # ═══════════════════════════════════════════════════════════════════

    @torch.no_grad()
    def warp(
        self,
        prev_frame: np.ndarray,
        curr_frame: np.ndarray,
        gaussian_model: nn.Module,
        is_delta: bool = False,
    ) -> dict:
        \"\"\"
        Run the full FR1-FR7 pipeline.

        Args:
            prev_frame: Previous frame as numpy HxWx3 uint8 RGB array
                        or PIL Image or torch Tensor [1,3,H,W].
            curr_frame: Current frame (same format as prev_frame).
            gaussian_model: The GaussianVideo_frame or GaussianVideo_delta model.
                           Must have `._xyz` parameter.
            is_delta: If True, the model is a GaussianVideo_delta and
                      positions are stored as deltas relative to p_xyz.

        Returns:
            dict with diagnostic info:
              - 'mean_flow': mean pixel displacement
              - 'clamped_pct': percentage of flow vectors that were clamped
              - 'num_gaussians': number of particles warped
        \"\"\"
        if not self.config.enabled:
            return {"skipped": True}

        cfg = self.config

        # ── Convert inputs to [1, 3, H, W] float32 tensors ────────────
        prev_t = self._to_tensor(prev_frame)   # [1, 3, H, W]
        curr_t = self._to_tensor(curr_frame)    # [1, 3, H, W]
        original_H, original_W = prev_t.shape[2], prev_t.shape[3]

        # ── FR1: Adaptive VRAM Protection ──────────────────────────────
        prev_ds, curr_ds, scale_factor = self._adaptive_downscale(
            prev_t, curr_t, original_H, original_W
        )

        # ── FR2: Eulerian Flow Extraction ──────────────────────────────
        flow = self._compute_flow(prev_ds, curr_ds)  # [1, 2, H_ds, W_ds]

        # ── FR3: Resolution Restoration & Safety Clamping ──────────────
        flow = self._restore_and_clamp(flow, original_H, original_W, scale_factor)
        # flow is now [1, 2, H_orig, W_orig]

        # ── FR4: Particle Space Normalization ──────────────────────────
        if is_delta:
            # For delta models: actual position = tanh(_xyz + p_xyz)
            # We need to advect the *combined* position
            raw_params = gaussian_model._xyz.data  # [N, 2]  (delta)
            p_xyz = gaussian_model.p_xyz            # [N, 2]  (parent buffer)
            combined_params = raw_params + p_xyz    # [N, 2]  (combined atanh space)
            normalized_pos = torch.tanh(combined_params)  # [N, 2] in [-1, 1]
        else:
            raw_params = gaussian_model._xyz.data   # [N, 2]
            normalized_pos = torch.tanh(raw_params) # [N, 2] in [-1, 1]

        N = normalized_pos.shape[0]

        # ── FR5: Sub-Pixel Eulerian-to-Lagrangian Mapping ──────────────
        per_particle_flow = self._sample_flow_at_particles(
            flow, normalized_pos, original_H, original_W
        )  # [N, 2]

        # ── FR6: Kinematic Advection ───────────────────────────────────
        advected_pos = self._advect(
            normalized_pos, per_particle_flow, original_H, original_W
        )  # [N, 2] in [-1, 1]

        # ── FR7: State Injection & Optimizer Prep ──────────────────────
        diagnostics = self._inject_state(
            gaussian_model, advected_pos, is_delta, per_particle_flow
        )
        diagnostics["num_gaussians"] = N

        if cfg.verbose:
            print(
                f"  [Turbo] N={N}, "
                f"mean_flow={diagnostics['mean_flow']:.2f}px, "
                f"clamped={diagnostics['clamped_pct']:.1f}%"
            )

        return diagnostics

    # ═══════════════════════════════════════════════════════════════════
    # FR1: Adaptive VRAM Protection
    # ═══════════════════════════════════════════════════════════════════

    def _adaptive_downscale(self, prev_t, curr_t, H, W):
        \"\"\"
        Downsample frames if they exceed max_resolution, ensuring
        dimensions are divisible by 8 (RAFT architecture constraint).

        Returns:
            (prev_ds, curr_ds, scale_factor)
        \"\"\"
        cfg = self.config
        scale_factor = 1.0

        if cfg.max_resolution > 0 and H > cfg.max_resolution:
            scale_factor = cfg.max_resolution / H
            new_H = int(H * scale_factor)
            new_W = int(W * scale_factor)
            # Round down to nearest multiple of 8
            new_H = (new_H // 8) * 8
            new_W = (new_W // 8) * 8
            # Recompute exact scale factor after rounding
            scale_factor = new_H / H

            prev_t = F.interpolate(
                prev_t, size=(new_H, new_W),
                mode="bilinear", align_corners=False
            )
            curr_t = F.interpolate(
                curr_t, size=(new_H, new_W),
                mode="bilinear", align_corners=False
            )
        else:
            # Still ensure divisible by 8
            new_H = (H // 8) * 8
            new_W = (W // 8) * 8
            if new_H != H or new_W != W:
                prev_t = F.interpolate(
                    prev_t, size=(new_H, new_W),
                    mode="bilinear", align_corners=False
                )
                curr_t = F.interpolate(
                    curr_t, size=(new_H, new_W),
                    mode="bilinear", align_corners=False
                )
                scale_factor = new_H / H

        return prev_t, curr_t, scale_factor

    # ═══════════════════════════════════════════════════════════════════
    # FR2: Eulerian Flow Extraction
    # ═══════════════════════════════════════════════════════════════════

    def _compute_flow(self, prev_ds, curr_ds):
        \"\"\"
        Run RAFT-Small on the (possibly downscaled) frame pair.

        Args:
            prev_ds: [1, 3, H_ds, W_ds] float32 tensor in [0, 1]
            curr_ds: [1, 3, H_ds, W_ds] float32 tensor in [0, 1]

        Returns:
            flow: [1, 2, H_ds, W_ds] — dense Eulerian flow field (dx, dy)
        \"\"\"
        cfg = self.config
        raft = _get_raft_model(self.device)

        # RAFT expects [0, 255] range input
        prev_input = (prev_ds * 255.0).to(self.device)
        curr_input = (curr_ds * 255.0).to(self.device)

        if cfg.use_fp16_flow and self.device.type == "cuda":
            with torch.cuda.amp.autocast():
                flow_list = raft(prev_input.half(), curr_input.half(),
                                 num_flow_updates=cfg.raft_iterations)
        else:
            flow_list = raft(prev_input, curr_input,
                             num_flow_updates=cfg.raft_iterations)

        # RAFT returns a list of iterative refinements; take the last
        flow = flow_list[-1].float()  # [1, 2, H_ds, W_ds]

        return flow

    # ═══════════════════════════════════════════════════════════════════
    # FR3: Resolution Restoration & Safety Clamping
    # ═══════════════════════════════════════════════════════════════════

    def _restore_and_clamp(self, flow, target_H, target_W, scale_factor):
        \"\"\"
        Upsample flow back to original resolution and clamp magnitude.

        The flow is in pixel displacement units at the downscaled resolution.
        After upscaling, we must divide by scale_factor to correct the
        magnitude to original-resolution pixel units.
        \"\"\"
        cfg = self.config

        # Upsample to original resolution
        if flow.shape[2] != target_H or flow.shape[3] != target_W:
            flow = F.interpolate(
                flow, size=(target_H, target_W),
                mode="bilinear", align_corners=False
            )

        # Correct magnitude: flow was computed at downscaled res
        if scale_factor != 1.0 and scale_factor > 0:
            flow = flow / scale_factor

        # Safety clamp
        if cfg.max_flow_magnitude > 0:
            flow_mag = torch.sqrt(flow[:, 0:1] ** 2 + flow[:, 1:2] ** 2)
            # Count how many exceed threshold (for diagnostics)
            self._clamped_count = (flow_mag > cfg.max_flow_magnitude).sum().item()
            self._total_pixels = flow_mag.numel()

            # Clamp by scaling down vectors that exceed max magnitude
            scale_mask = torch.clamp(
                cfg.max_flow_magnitude / (flow_mag + 1e-8), max=1.0
            )
            flow = flow * scale_mask
        else:
            self._clamped_count = 0
            self._total_pixels = flow.shape[2] * flow.shape[3]

        return flow

    # ═══════════════════════════════════════════════════════════════════
    # FR5: Sub-Pixel Eulerian-to-Lagrangian Mapping
    # ═══════════════════════════════════════════════════════════════════

    def _sample_flow_at_particles(self, flow, normalized_pos, H, W):
        \"\"\"
        Use grid_sample to probe the Eulerian flow field at each
        Gaussian particle's sub-pixel position.

        Args:
            flow:           [1, 2, H, W] flow field in pixel units
            normalized_pos: [N, 2] positions in [-1, 1] (x, y)
            H, W:           original image dimensions

        Returns:
            per_particle_flow: [N, 2] — individualized (dx, dy) in pixel units
        \"\"\"
        N = normalized_pos.shape[0]

        # grid_sample expects grid in shape [1, 1, N, 2] with (x, y) in [-1,1]
        # Our normalized_pos is already in [-1, 1] (column0=x, column1=y)
        grid = normalized_pos.unsqueeze(0).unsqueeze(0)  # [1, 1, N, 2]

        # grid_sample: input [1, 2, H, W], grid [1, 1, N, 2]
        # Output: [1, 2, 1, N]
        sampled = F.grid_sample(
            flow, grid, mode="bilinear",
            padding_mode="border", align_corners=True
        )  # [1, 2, 1, N]

        # Reshape to [N, 2]
        per_particle_flow = sampled.squeeze(0).squeeze(1).permute(1, 0)  # [N, 2]

        return per_particle_flow

    # ═══════════════════════════════════════════════════════════════════
    # FR6: Kinematic Advection
    # ═══════════════════════════════════════════════════════════════════

    def _advect(self, normalized_pos, per_particle_flow, H, W):
        \"\"\"
        Translate particles by their sampled flow vectors.

        The flow is in pixel units, but positions are in [-1, 1] normalized
        space. We convert flow to normalized units by dividing by (W/2, H/2).

        Args:
            normalized_pos:    [N, 2] in [-1, 1] (x, y)
            per_particle_flow: [N, 2] in pixel units (dx, dy)
            H, W:              image dimensions

        Returns:
            advected_pos: [N, 2] in [-1, 1]
        \"\"\"
        # Convert pixel displacement to normalized displacement
        # x-axis: divide by W/2 ;  y-axis: divide by H/2
        norm_flow = per_particle_flow.clone()
        norm_flow[:, 0] = per_particle_flow[:, 0] / (W / 2.0)  # dx
        norm_flow[:, 1] = per_particle_flow[:, 1] / (H / 2.0)  # dy

        advected_pos = normalized_pos + norm_flow

        return advected_pos

    # ═══════════════════════════════════════════════════════════════════
    # FR7: State Injection & Optimizer Prep
    # ═══════════════════════════════════════════════════════════════════

    def _inject_state(self, gaussian_model, advected_pos, is_delta, per_particle_flow):
        \"\"\"
        Clamp, apply inverse transform (atanh), and overwrite _xyz.

        For delta models, we subtract p_xyz before writing back so that
        the delta semantics are preserved.

        Returns diagnostics dict.
        \"\"\"
        cfg = self.config

        # Compute diagnostics before clamping
        flow_magnitude = torch.sqrt(
            per_particle_flow[:, 0] ** 2 + per_particle_flow[:, 1] ** 2
        )
        mean_flow = flow_magnitude.mean().item()

        # Strict boundary clamping
        advected_clamped = torch.clamp(
            advected_pos, -cfg.position_clamp, cfg.position_clamp
        )

        # Inverse transform: normalized [-1,1] → unconstrained atanh space
        new_atanh = torch.atanh(advected_clamped)

        if is_delta:
            # For delta model: _xyz stores the *delta*, not the full position
            # full_atanh = delta + p_xyz  →  delta = full_atanh - p_xyz
            new_delta = new_atanh - gaussian_model.p_xyz
            gaussian_model._xyz.data.copy_(new_delta)
        else:
            gaussian_model._xyz.data.copy_(new_atanh)

        # Reset optimizer momentum for _xyz to prevent stale gradients
        # from fighting the advection
        self._reset_optimizer_state(gaussian_model)

        clamped_pct = (
            self._clamped_count / max(self._total_pixels, 1) * 100.0
        )

        return {
            "mean_flow": mean_flow,
            "clamped_pct": clamped_pct,
        }

    def _reset_optimizer_state(self, gaussian_model):
        \"\"\"
        Reset the optimizer's momentum buffers for _xyz parameter only.
        This prevents stale momentum from the previous frame from
        counteracting the flow-based advection.
        \"\"\"
        if not hasattr(gaussian_model, 'optimizer'):
            return

        optimizer = gaussian_model.optimizer

        for group in optimizer.param_groups:
            for p in group['params']:
                if p is gaussian_model._xyz:
                    if p in optimizer.state:
                        state = optimizer.state[p]
                        # Reset all momentum buffers
                        for key in list(state.keys()):
                            if isinstance(state[key], torch.Tensor) and key != 'step':
                                state[key].zero_()
                    break

    # ═══════════════════════════════════════════════════════════════════
    # Utility
    # ═══════════════════════════════════════════════════════════════════

    def _to_tensor(self, frame):
        \"\"\"
        Convert various frame formats to [1, 3, H, W] float32 tensor on device.
        Supports: numpy uint8 HWC, PIL Image, torch Tensor.
        \"\"\"
        if isinstance(frame, torch.Tensor):
            t = frame.float()
            if t.dim() == 3:
                t = t.unsqueeze(0)  # [3,H,W] → [1,3,H,W]
            if t.max() > 1.0:
                t = t / 255.0
            return t.to(self.device)

        if isinstance(frame, Image.Image):
            frame = np.array(frame)

        if isinstance(frame, np.ndarray):
            # Expect HxWx3 uint8
            t = torch.from_numpy(frame).float() / 255.0
            t = t.permute(2, 0, 1).unsqueeze(0)  # [1, 3, H, W]
            return t.to(self.device)

        raise TypeError(f"Unsupported frame type: {type(frame)}")

""")

# Verify syntax
import ast
for fname in ["turbo_config.py", "optical_flow_warper.py"]:
    with open(f"/kaggle/working/GSVC/{fname}") as fh:
        ast.parse(fh.read())
    print(f"  {fname}: OK")
print("Setup complete.")


## T1 — Baseline (PRD §5 Test 1)

Original GSVC — no optical flow pre-advection.  
Establishes reference **BPP, PSNR, and encoding time** for comparison.


In [ ]:
# ============================================================
# T1: BASELINE — No turbo, pure gradient-descent tracking
# ============================================================
%cd /kaggle/working/GSVC

!python3 train_video_Represent.py \
  --loss_type L2 \
  --dataset /kaggle/input/models/vcoklat/beauty/other/default/1/Beauty_1920x1080_120fps_420_8bit_YUV.yuv \
  --data_name Beauty \
  --num_points 10000 \
  --iterations 100000 \
  --is_rm --is_ad \
  --image_length 10 \
  --savdir ablation_T1_baseline \
  --savdir_m ablation_T1_baseline_models


## T2 — Full Turbo Pipeline (PRD §5 Test 2)

All FR1-FR7 enabled (`--turbo --turbo_preset default`).  
Measures **iteration reduction** and **RD improvement** vs baseline.


In [ ]:
# ============================================================
# T2: FULL TURBO — All FR1-FR7 active (default preset)
#   FR1: downscale to ≤540p for RAFT
#   FR2: RAFT-Small optical flow
#   FR3: upsample + 80px magnitude clamp
#   FR4: tanh normalization
#   FR5: grid_sample sub-pixel E→L mapping
#   FR6: kinematic advection
#   FR7: atanh injection + optimizer reset
# ============================================================
%cd /kaggle/working/GSVC

!python3 train_video_Represent.py \
  --loss_type L2 \
  --dataset /kaggle/input/models/vcoklat/beauty/other/default/1/Beauty_1920x1080_120fps_420_8bit_YUV.yuv \
  --data_name Beauty \
  --num_points 10000 \
  --iterations 100000 \
  --is_rm --is_ad \
  --image_length 10 \
  --savdir ablation_T2_turbo \
  --savdir_m ablation_T2_turbo_models \
  --turbo \
  --turbo_preset default \
  --turbo_verbose


## T3 — Clamping Ablation (PRD §5 Test 3)

**Disables FR3 (flow magnitude clamp) and FR7 (position clamp).**  
Expected: some out-of-bounds splats → PSNR degradation and possible instability.
- `max_flow_magnitude = 0.0` → no clamping applied to flow vectors
- `position_clamp = 1.0` → atanh receives values that may → ∞


In [ ]:
# ============================================================
# T3: CLAMPING ABLATION — FR3 + FR7 disabled
# ============================================================
%cd /kaggle/working/GSVC

!python3 train_video_Represent.py \
  --loss_type L2 \
  --dataset /kaggle/input/models/vcoklat/beauty/other/default/1/Beauty_1920x1080_120fps_420_8bit_YUV.yuv \
  --data_name Beauty \
  --num_points 10000 \
  --iterations 100000 \
  --is_rm --is_ad \
  --image_length 10 \
  --savdir ablation_T3_no_clamp \
  --savdir_m ablation_T3_no_clamp_models \
  --turbo \
  --turbo_preset ablation_no_clamp \
  --turbo_verbose


## T4 — VRAM Limit Ablation (PRD §5 Test 4)

**Disables FR1 (adaptive downscaling).**  
RAFT receives native 1080p input.  
Documents VRAM spike and any encoding-time increase to justify the 540p downscaling design.

> ⚠️ This may hit OOM on Kaggle's T4 (16 GB). Watch the VRAM usage.


In [ ]:
# ============================================================
# T4: VRAM ABLATION — FR1 disabled, native 1080p to RAFT
# ============================================================
%cd /kaggle/working/GSVC

!python3 train_video_Represent.py \
  --loss_type L2 \
  --dataset /kaggle/input/models/vcoklat/beauty/other/default/1/Beauty_1920x1080_120fps_420_8bit_YUV.yuv \
  --data_name Beauty \
  --num_points 10000 \
  --iterations 100000 \
  --is_rm --is_ad \
  --image_length 10 \
  --savdir ablation_T4_no_downscale \
  --savdir_m ablation_T4_no_downscale_models \
  --turbo \
  --turbo_preset ablation_no_downscale \
  --turbo_verbose


## Results — PRD Ablation Comparison

Reads `train.txt` from each test run and prints a unified comparison table.


In [ ]:
# ============================================================
# RESULTS: Compare all 4 ablation test configs
# ============================================================
import re
from pathlib import Path

TESTS = {
    "T1 Baseline":        "ablation_T1_baseline",
    "T2 Full Turbo":      "ablation_T2_turbo",
    "T3 No Clamp":        "ablation_T3_no_clamp",
    "T4 No Downscale":    "ablation_T4_no_downscale",
}

def parse_log(savdir):
    pattern = (Path("/kaggle/working/GSVC/checkpoints")
               / savdir / "Beauty" / "GaussianVideo_100000_10000" / "train.txt")
    if not pattern.exists():
        return None
    lines = pattern.read_text().splitlines()
    results = []
    for line in lines:
        m = re.search(
            r'Frame_(\d+):.*PSNR:([\d.]+).*MS-SSIM:([\d.]+).*Training:([\d.]+)',
            line
        )
        if m:
            results.append({
                'frame': int(m.group(1)),
                'psnr':  float(m.group(2)),
                'msssim':float(m.group(3)),
                'time':  float(m.group(4)),
            })
    return results or None

print(f"{'Config':<22} {'AvgPSNR':>9} {'MinPSNR':>9} {'AvgTime':>9} {'<30dB':>6}  Frames")
print("-" * 72)

all_results = {}
for name, savdir in TESTS.items():
    rows = parse_log(savdir)
    if rows is None:
        print(f"{name:<22}  (no log found — run this test first)")
        continue
    psnrs = [r['psnr'] for r in rows]
    times = [r['time'] for r in rows]
    below = sum(1 for p in psnrs if p < 30.0)
    all_results[name] = rows
    print(f"{name:<22} {sum(psnrs)/len(psnrs):>9.4f} {min(psnrs):>9.4f} "
          f"{sum(times)/len(times):>9.2f}s {below:>6}  "
          f"{[r['frame'] for r in rows if r['psnr']<30.0] or '✓ all ≥30'}")

if len(all_results) >= 2:
    base = all_results.get("T1 Baseline")
    turbo = all_results.get("T2 Full Turbo")
    if base and turbo:
        base_psnrs  = {r['frame']: r['psnr'] for r in base}
        turbo_psnrs = {r['frame']: r['psnr'] for r in turbo}
        base_times  = {r['frame']: r['time'] for r in base}
        turbo_times = {r['frame']: r['time'] for r in turbo}
        frames = sorted(set(base_psnrs) & set(turbo_psnrs))
        psnr_gain = sum(turbo_psnrs[f] - base_psnrs[f] for f in frames) / len(frames)
        time_gain = sum(base_times[f] - turbo_times[f] for f in frames) / len(frames)
        pct_faster = time_gain / (sum(base_times[f] for f in frames) / len(frames)) * 100
        print()
        print(f"T2 vs T1 — Avg PSNR gain : {psnr_gain:+.4f} dB")
        print(f"T2 vs T1 — Avg time saved: {time_gain:+.2f}s/frame  ({pct_faster:+.1f}%)")
        if pct_faster >= 50:
            print("  ✓ KPI met: >50% faster than baseline")
        else:
            print(f"  ✗ KPI not met yet ({pct_faster:.1f}% < 50% target)")
